# Single-cell perturbations: split and baseline workflow

This notebook is the shared entry point for data splitting and future baseline modules.

**Version 1:** use `T cells CD4+` as the pseudo-target, keep **17** of its cell–drug pairs in training, and hold out its remaining pairs for validation. Keep every available pair from other cell types in training. Use **seed 42**. Sampling includes control pairs; no additional control filtering is applied.

All baselines must reuse the saved pair IDs. This evaluates missing combinations of known drugs and cell types, rather than completely unseen drugs. Only `de_train.parquet` and the small `id_map.csv` are loaded; raw single-cell matrices are not needed.

Dependencies: `numpy`, `pandas`, `pyarrow`, and a Jupyter kernel (the existing `jhu_bds` environment supports these).

In [ ]:
from pathlib import Path
import sys

# Works when launched from Singlecell_perturbations or the repository root.
PROJECT_DIR = Path.cwd().resolve()
if not (PROJECT_DIR / "split_data.py").is_file():
    PROJECT_DIR = PROJECT_DIR / "Singlecell_perturbations"
if not (PROJECT_DIR / "split_data.py").is_file():
    raise FileNotFoundError("Open this notebook from Singlecell_perturbations or the repo root.")
sys.path.insert(0, str(PROJECT_DIR))
sys.dont_write_bytecode = True

import pandas as pd
from IPython.display import display
from split_data import load_or_create_split, apply_split, get_gene_columns

DATA_DIR = PROJECT_DIR / "open-problems-single-cell-perturbations"
TARGET_CELL_TYPE = "T cells CD4+"
N_OBSERVED = 17
RANDOM_SEED = 42
# Use a new filename if intentionally changing the experiment settings.
SPLIT_PATH = PROJECT_DIR / "splits" / "cd4_observed17_seed42.json"

## 1. Load labeled DE data

Each row is one cell-type–drug pair. The gene columns are **DE targets**, not input features. The official test pair list is kept separate from internal validation.

In [ ]:
de_train = pd.read_parquet(DATA_DIR / "de_train.parquet")
id_map = pd.read_csv(DATA_DIR / "id_map.csv")
gene_cols = get_gene_columns(de_train)

print(f"Labeled pairs: {len(de_train)}")
print(f"DE target genes: {len(gene_cols)}")
print(f"Official test pairs: {len(id_map)}")
display(de_train.groupby("cell_type").size().rename("available_pairs").to_frame())

## 2. Create or reload the fixed split

The first run saves a small JSON manifest containing settings and `(cell_type, sm_name)` IDs. Later runs reload that manifest, even if the source row order changes. A mismatch in settings or available pairs raises an error instead of silently changing the experiment. Keep this small manifest with the project so all models and collaborators use the same split; it contains no DE matrix.

In [ ]:
split = load_or_create_split(
    de_train, SPLIT_PATH,
    target_cell_type=TARGET_CELL_TYPE,
    n_observed=N_OBSERVED,
    seed=RANDOM_SEED,
)
train_df, val_df = apply_split(de_train, split)

summary = pd.concat([
    train_df.groupby("cell_type").size().rename("train"),
    val_df.groupby("cell_type").size().rename("validation"),
], axis=1).fillna(0).astype(int)
summary["total"] = summary.sum(axis=1)
display(summary)
print(f"Train: {len(train_df)} | Validation: {len(val_df)}")
print(f"Saved split: {SPLIT_PATH.relative_to(PROJECT_DIR)}")

In [ ]:
train_pairs = set(train_df[["cell_type", "sm_name"]].itertuples(index=False, name=None))
val_pairs = set(val_df[["cell_type", "sm_name"]].itertuples(index=False, name=None))
test_pairs = set(id_map[["cell_type", "sm_name"]].itertuples(index=False, name=None))

assert train_pairs.isdisjoint(val_pairs)
assert len(train_pairs | val_pairs) == len(de_train)
assert val_df["cell_type"].eq(TARGET_CELL_TYPE).all()
assert train_df["cell_type"].eq(TARGET_CELL_TYPE).sum() == N_OBSERVED
assert set(val_df["sm_name"]).issubset(set(train_df["sm_name"]))
assert (train_pairs | val_pairs).isdisjoint(test_pairs)
print("Checks passed: complete partition, no pair overlap, validation drugs seen in training.")

## 3. Shared inputs for future baseline modules

`X_train` / `X_val` contain the cell and drug identifiers. `y_train` / `y_val` contain the DE vectors in exactly the same gene order. Dataframes retain aligned source indices; predictions should follow the row order of `X_val`.

Fit encoders, scalers, PCA, and models using the **training subset only** during model development. Never use validation DE values to construct training features. If adding raw-data features later, exclude held-out treated cell–drug combinations from those features as well.

In [ ]:
feature_cols = ["cell_type", "sm_name"]
X_train = train_df[feature_cols].copy()
X_val = val_df[feature_cols].copy()
y_train = train_df[gene_cols].copy()
y_val = val_df[gene_cols].copy()
X_test = id_map[feature_cols].copy()
test_ids = id_map["id"].copy()

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val:  ", X_val.shape, "y_val:  ", y_val.shape)
print("X_test: ", X_test.shape)
assert X_train.index.equals(y_train.index)
assert X_val.index.equals(y_val.index)
assert y_train.columns.equals(y_val.columns)

## 4. Baseline calls — add here later

No baseline module exists yet. Future modules can expose a function such as `fit_predict(X_train, y_train, X_val)` and return a prediction matrix matching `y_val.shape` and `gene_cols` order. Compare all baselines on this same split using MRRMSE.

The commented example below is an interface proposal, not a currently available module.

In [ ]:
# Future example once baseline_ridge.py implements this interface:
# from baseline_ridge import fit_predict
# val_predictions = fit_predict(X_train, y_train, X_val)
# assert val_predictions.shape == y_val.shape

## 5. Final training — after model selection

Once model choices and hyperparameters are fixed, refit the entire pipeline (including PCA, if used) on **all 614 labeled pairs**, then predict the **255 official test pairs** in `id_map` order. Keep the submission gene order consistent with `sample_submission.csv`. The internal validation subset is used for development; it is included again in final training.

Multiple pseudo-target splits and B/Myeloid cross-validation are possible later; they are not part of this first split.